In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-04-21 14:13:45.483256: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [4]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]
CHANNEL="ee"
# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]

# # for dev
# INPUTDIR = setting["INPUTDIR"]+'/ee18/'
# INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20230419/Presel/ee16post/'
# REGION = 'SR'

2803_equalpropv1


# Load data

In [5]:
if REGION == 'SR':
    file = 'nested_dic_df.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 

# # temp!!
for key in dic_df_met_u_unsort:
    df = dic_df_met_u_unsort[key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_u_unsort[key] = df.drop(i)
    df = dic_df_met_d_unsort[key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_d_unsort[key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]

Zp-4100_CH-1345
DY1500To2000
Zp-1700_CH-345
Zp-2500_CH-345
DY500To700
Zp-1700_CH-845
Zp-4100_CH-595
Zp-3300_CH-345
Zp-1700_CH-595
Zp-3700_CH-345
Zp-2100_CH-595
Zp-2900_CH-345
Zp-4100_CH-1095
Zp-2900_CH-845
Zp-3700_CH-1345
Zp-3300_CH-1595
Zp-2500_CH-595
Zp-4100_CH-345
Zp-3700_CH-1845
Zp-2500_CH-845
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-3300_CH-595
Zp-2100_CH-345
Zp-2900_CH-595
Zp-4100_CH-845
Zp-2100_CH-845
Zp-3300_CH-1345
Zp-3700_CH-1595
DY2000To3000
Zp-3700_CH-595
Zp-2900_CH-1345
Zp-3700_CH-1095
DY800To1000
DY1000To1500
Zp-3700_CH-845
Zp-3300_CH-1095
Zp-2900_CH-1095
TT
TTSemi
TTZ
DY
top
antitop
ZZ
WZ
WW
Data


In [6]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'MET_pre', 'MET_phi_pre',
       'PuppiMET_phi_pre', 'eta_lead_pre', 'eta_trail_pre', 'phi_lead_pre',
       'phi_trail_pre', 'njet_pre', 'nbjetflav_pre', 'genweight_u_pre',
       'genweight_d_pre', 'genweight_pre', 'puweight_pre', 'puweight_u_pre',
       'puweight_d_pre', 'l1prefiring_pre', 'l1prefiring_u_pre',
       'l1prefiring_d_pre', 'effSF_pre', 'effSF_u_pre', 'effSF_d_pre',
       'trigSF_pre', 'trigSF_u_pre', 'trigSF_d_pre', 'topSF_pre',
       'topSF_u_pre', 'topSF_d_pre', 'btagSF_pre', 'weight', 'wopu', 'wol1',
       'woeff', 'wotop', 'wobsf', 'weightpuweightUp', 'weightpuweightDown',
       'weightl1prefiringUp', 'weightl1prefiringDown', 'weighteffSFUp',
       'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDown',
       'weightgenweightUp', 'weightgenweightDown', 'weighttrigSFUp',


In [7]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [8]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [9]:
import math
sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.3
else:
    threshold = 1    
    
def draw_with_systematics(model_mass, dic_df_raw, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/norm_dnnscore.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']
    hist.create_root_file(file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    for s in sys_list:
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [10]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
    sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.3
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass)     
    hist.export_yields_table(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [18]:
import time

nbin=20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
# model_masses = ['Zp-2500_CH-845']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = INPUTDIR

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d = [{} for _ in range(3)]

# masses loop
for model_mass in model_masses:
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]
    
#     file_postfix = 'test'

    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d = [{} for _ in range(3)]        

    for key2 in dic_df_train:
        dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
        dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
        if dic_df_train[key2].empty:
            continue
        dic_df_scale[key2] = Transformer.transform(dic_df_train[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d = [{} for _ in range(3)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort = [{} for _ in range(3)]

for key4 in dic_df_scale:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weightmetUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weightmetDown"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d
    
for model_mass in model_masses:
    for key5 in dic_df_scale:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]
    
        
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1682080221.0730462


/tmp/ipykernel_616/3047493747.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_616/3047493747.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_616/3047493747.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_616/3047493747.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
/tmp/ipykernel_616/3047493747.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_616/3047493747.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_616/3047493747.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_616/3047493747.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_616/3047493747.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_616/3047493747.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_616/3047493747.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_616/3047493747.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

...

        0.55732662,  0.42349637,  0.44090897,  0.44838786,  0.34381819])}
{'puweightUp': array([10.24529797, -0.22639235,  0.69172155, -0.68683552,  0.05048476,
        0.05374136,  0.04879198, -0.07779984,  0.22693887,  0.12279152,
        0.14176771,  0.71112781,  0.55728139, -0.02941934,  0.90823584,
        0.68759324,  0.19557169, -0.01661897,  0.26014473,  0.07351764]), 'puweightDown': array([-8.54520556e+00, -1.45842314e-01, -6.95387024e-01,  5.96887725e-01,
        1.33476483e-02, -7.26358090e-02, -1.36540644e-01,  9.58950074e-02,
       -2.88399540e-01, -1.97422236e-01, -1.90981337e-01, -8.53388173e-01,
       -4.43509367e-01, -8.04815585e-03, -7.96848105e-01, -6.26166268e-01,
       -1.93181294e-01,  4.87507932e-02, -2.78830931e-01, -1.24150206e-01]), 'l1prefiringUp': array([19.67575862,  2.80444884,  1.23484843,  0.65698814,  0.36177417,
        0.12153958,  0.26319895,  0.1261043 ,  0.15239664,  0.20623316,
        0.08437382,  0.12173868,  0.29394632,  0.12822161,  0.11138

        0.88758211,  0.55701857,  0.51316067,  0.52908821,  0.47823068])}
{'puweightUp': array([ 1.10631997e+01, -3.54425059e-01, -5.35260489e-02, -5.62496420e-01,
        1.16908128e-02,  9.13334727e-01,  1.15934925e+00,  4.71351206e-01,
        2.76249296e-02,  6.88250033e-01, -6.18536220e-02, -2.68069114e-01,
        7.42582836e-01, -3.04540067e-04, -4.53893500e-02, -5.14892567e-02,
        5.71707382e-02,  4.15508185e-02,  1.33724135e-01,  2.56662673e-02]), 'puweightDown': array([-9.63918189e+00,  3.44011651e-02,  1.41223367e-01,  5.85260539e-01,
       -1.82244409e-02, -1.06947681e+00, -1.06982018e+00, -5.47421075e-01,
       -5.52075896e-02, -6.34447063e-01,  7.10149440e-02,  1.67630982e-01,
       -5.18188681e-01,  6.46621067e-04, -3.45918864e-02,  3.08715718e-02,
       -7.20150743e-02, -8.57825808e-03, -1.40684152e-01, -6.48678725e-02]), 'l1prefiringUp': array([21.14070513,  2.26151148,  0.5757706 ,  0.53049983,  0.25160349,
        0.28180531,  0.38684257,  0.13169796,  0.128

        0.53234451,  0.46216752,  0.43814526,  0.43238281,  0.40088463])}
{'puweightUp': array([ 9.40502790e+00,  1.21345076e+00, -1.79207296e-01, -1.08659656e-01,
       -4.95061167e-01,  2.27998137e-02,  3.42963513e-01,  3.41266701e-01,
        1.34298164e-01,  4.60599068e-01,  1.82761269e-01,  4.72400324e-01,
        3.88577413e-01,  4.88285825e-01,  9.08345433e-01,  6.90883492e-03,
       -7.10208097e-02,  1.29490338e-01,  1.77531130e-01,  1.17184468e-01]), 'puweightDown': array([-8.07313842, -1.10659236, -0.01072608,  0.0533814 ,  0.60829591,
       -0.0780638 , -0.40146437, -0.35426   , -0.16960152, -0.58703789,
       -0.29804906, -0.41554098, -0.42221195, -0.23050359, -0.96737936,
        0.01001155,  0.05355099, -0.11883087, -0.18192847, -0.15156694]), 'l1prefiringUp': array([20.48753611,  2.47417156,  1.06129315,  0.49236068,  0.18550883,
        0.33304021,  0.1395622 ,  0.20384194,  0.1098672 ,  0.13204704,
        0.10505374,  0.30254727,  0.11066196,  0.11727732,  0.20418

        0.58855446,  0.42387929,  0.47439042,  0.4493977 ,  0.42579297])}
array('f', [5.269317626953125, 5.806262016296387, 11.392960548400879, 135.41049194335938, 302.4034729003906, 5.45914363861084, 2845.911376953125, 95.21288299560547, 6.208733558654785])
test
array('f', [5.269317626953125, 5.806262016296387, 11.392960548400879, 135.41049194335938, 302.4034729003906, 5.45914363861084, 2845.911376953125, 95.21288299560547, 6.208733558654785])
test
array('f', [5.269317626953125, 5.806262016296387, 11.392960548400879, 135.41049194335938, 302.4034729003906, 5.45914363861084, 2845.911376953125, 95.21288299560547, 6.208733558654785])
test
array('f', [5.269317626953125, 5.806262016296387, 11.392960548400879, 135.41049194335938, 302.4034729003906, 5.45914363861084, 2845.911376953125, 95.21288299560547, 6.208733558654785])
test


Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-3300_CH-1345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-3300_CH-1345.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-3700_CH-595breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-3700_CH-595.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-1700_CH-595breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2100_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2500_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-4100_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-4100_CH-1345breakdown.pdf has be